> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# WINFUT intraday momentum: analysis notebook

The strategy of Zarattini, Aziz and Barbon, *Beat the Market* (SPY), as coded in the authors' notebook, run on the **WIN mini Ibovespa future** (`WIN$N`, 1-minute bars, R$ 0.20 a point). The rules and every decision are in `docs/brief.md` section 10.

**Read this first**
- **Development data only.** The loader returns bars before 2025-01-01; the final test period is **not** loaded, and the notebook stops if a later bar shows up. The owner decides when the final test runs.
- **Every number you read here is a look at the development data.** The brief keeps a count (five so far). Trying many settings and keeping the best one is overfitting: the search over start and exit times belongs inside the pre-registered plan, not in this notebook. Use it to *understand*, not to *pick*.
- **What the strategy does.** Bands from the mean absolute move from the open at the same time of day over the previous 14 days, widened by the overnight gap. Every 30 minutes: long if the price is above the upper band **and** the VWAP, short if below the lower band **and** the VWAP, flat otherwise. A position is held until a later decision says flat, or until the exit time. Size is set once at the open of each day (volatility target, or a fixed number of contracts).
- **The three drawdowns.** *Balance* is the end-of-day result (the strategy is flat every night). *Equity* marks the open position at every 1-minute close. *Adverse* uses each bar's low for a long and high for a short, the worst the account could have been inside the minute. Nothing here sees inside a minute.
- **Not modelled:** slippage (unless you set ticks below), the closing-call mechanics, taxes, margin calls, a roll-day correction (the series is unadjusted: roll days show a +1% to +2.7% overnight gap).

In [ ]:
import warnings
from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from b3core.intraday_momentum import (
    IntradayConfig,
    breakout_signal,
    drawdown_summary,
    intraday_equity,
    noise_bands,
    performance,
    run_intraday_momentum,
    session_vwap,
    strategy_exposure,
)
from b3core.noise_area import noise_sigma
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200

## 1. Parameters

Change these and re-run from section 2. `SIZING = "fixed"` trades `FIXED_CONTRACTS` every day (1 is the single-contract version); `"vol_target"` is the paper's volatility target. Times are `HH:MM`. `SLIPPAGE_TICKS` adds that many ticks (5 points, R$ 1.00) per contract and side on top of the B3 fee.

In [ ]:
SIZING = "fixed"  # "fixed" or "vol_target"
FIXED_CONTRACTS = 1
FIRST_DECISION = "09:00"  # the earliest time a decision may be taken (the first possible one is 09:30)
EXIT = "18:20"  # the time the position is flat; the last bar of the session closes at 18:25
B3_FEE_PER_SIDE = 0.25  # R$ per contract and side: the owner's R$ 0.50 round trip, no brokerage
SLIPPAGE_TICKS = 0.0
TICK_REAIS = 1.0  # 5 points x R$ 0.20
CAPITAL = 100_000.0
MARGIN_PER_CONTRACT = 1_000.0  # the broker's margin for one WIN mini contract


def minutes(hhmm: str) -> int:
    hour, minute = hhmm.split(":")
    return int(hour) * 60 + int(minute)


def make_config(**changes) -> IntradayConfig:
    settings = dict(
        first_decision=minutes(FIRST_DECISION),
        exit_minute=minutes(EXIT),
        fee_per_side=B3_FEE_PER_SIDE + SLIPPAGE_TICKS * TICK_REAIS,
        capital=CAPITAL,
        margin_per_contract=MARGIN_PER_CONTRACT,
        fixed_contracts=FIXED_CONTRACTS if SIZING == "fixed" else None,
    )
    settings.update(changes)
    return IntradayConfig(**settings)


config = make_config()
config

## 2. Load the development bars

In [ ]:
bars = load_bars("../samples")  # development period only: final=False is the default
assert bars.index.max() < pd.Timestamp(FINAL_START.date()), "a bar from the final test period was loaded"
day = bars.index.normalize()
print(f"{len(bars):,} one-minute bars, {day.nunique()} days, {bars.index[0].date()} to {bars.index[-1].date()}; ingest issues: {bars.attrs['issues'] or 'none'}")

last_bar = bars.groupby(day).apply(lambda g: g.index[-1].strftime("%H:%M"))
first_bar = bars.groupby(day).apply(lambda g: g.index[0].strftime("%H:%M"))
pd.DataFrame({"last bar of the day": last_bar.value_counts(), "first bar of the day": first_bar.value_counts()}).fillna(0).astype(int)

## 3. Run the strategy

One row per day: `contracts`, `trades` (sides), `gross`, `cost`, `net` (R$), `ret` (over the equity of the day before), `aum`, `margin`. The first day and the first days without a sigma do not trade.

In [ ]:
result = run_intraday_momentum(bars, config)
curve = intraday_equity(bars, config, result)  # equity and adverse equity at every bar
traded = result[result["ret"].notna()]
print(f"{len(traded)} days traded; {int((traded['trades'] > 0).sum())} with a position; "
      f"contracts a day: mean {traded['contracts'].mean():.1f}, max {traded['contracts'].max()}; "
      f"margin tied up: max R$ {traded['margin'].max():,.0f}")
display(traded.tail(8))

In [ ]:
perf = performance(result["ret"])
dd = drawdown_summary(result, curve, config.capital)
years = perf["days"] / 252
summary = pd.Series(
    {
        "net result, R$": traded["net"].sum(),
        "R$ a year": traded["net"].sum() / years,
        "final equity, R$": result["aum"].iloc[-1],
        "total return, %": perf["total_return"] * 100,
        "Sharpe (daily returns)": perf["sharpe"],
        "t-statistic of the mean": perf["t_stat"],
        "days with a gain, % of days with a result": perf["hit_ratio"] * 100,
        "trades (sides) a day": traded["trades"].mean(),
        "costs, R$ in total": traded["cost"].sum(),
        "balance drawdown, R$": dd["balance_drawdown"],
        "equity drawdown, R$ (bar closes)": dd["equity_drawdown"],
        "adverse drawdown, R$ (bar lows/highs)": dd["adverse_drawdown"],
        "worst day, R$ (end of day)": dd["worst_day"],
        "worst intraday loss, R$ (from the day's start)": dd["worst_intraday_loss"],
    },
    name="this run",
)
display(summary.to_frame())

## 4. Variants side by side

The same bars under different exits, sizings and costs. Reading this table is itself a look at the development data; it is here so the effect of each choice is visible, not so that one is picked from it. `1 contract` reports in R$ per contract; the account rows size by the volatility target on R$ 100,000.

In [ ]:
def evaluate(cfg: IntradayConfig) -> dict:
    res = run_intraday_momentum(bars, cfg)
    cur = intraday_equity(bars, cfg, res)
    done = res[res["ret"].notna()]
    stats = performance(res["ret"])
    draw = drawdown_summary(res, cur, cfg.capital)
    return {
        "net R$": done["net"].sum(),
        "R$ / year": done["net"].sum() / (stats["days"] / 252),
        "Sharpe": stats["sharpe"],
        "t": stats["t_stat"],
        "balance DD": draw["balance_drawdown"],
        "equity DD": draw["equity_drawdown"],
        "adverse DD": draw["adverse_drawdown"],
        "worst day": draw["worst_day"],
        "worst intraday": draw["worst_intraday_loss"],
        "contracts (avg)": done["contracts"].mean(),
    }


tick = TICK_REAIS
one = dict(fixed_contracts=1)
no_exit = minutes("24:00")
variants = {
    "1 contract, exit 18:20": make_config(**one, exit_minute=minutes("18:20"), fee_per_side=B3_FEE_PER_SIDE),
    "1 contract, exit 18:20, +1 tick": make_config(**one, exit_minute=minutes("18:20"), fee_per_side=B3_FEE_PER_SIDE + tick),
    "1 contract, exit 18:20, +2 ticks": make_config(**one, exit_minute=minutes("18:20"), fee_per_side=B3_FEE_PER_SIDE + 2 * tick),
    "1 contract, no exit": make_config(**one, exit_minute=no_exit, fee_per_side=B3_FEE_PER_SIDE),
    "1 contract, exit 17:30": make_config(**one, exit_minute=minutes("17:30"), fee_per_side=B3_FEE_PER_SIDE),
    "account (vol target), exit 18:20": make_config(fixed_contracts=None, exit_minute=minutes("18:20"), fee_per_side=B3_FEE_PER_SIDE),
    "account (vol target), exit 18:20, +1 tick": make_config(fixed_contracts=None, exit_minute=minutes("18:20"), fee_per_side=B3_FEE_PER_SIDE + tick),
    "account (vol target), exit 17:30": make_config(fixed_contracts=None, exit_minute=minutes("17:30"), fee_per_side=B3_FEE_PER_SIDE),
}
table = pd.DataFrame({name: evaluate(cfg) for name, cfg in variants.items()}).T
display(table.round(2))

## 5. Equity and drawdowns

The **balance** line is the end-of-day equity. The shaded band is each day's range of the account inside the day: the lowest *adverse* equity and the highest bar-close equity. The lower chart is the drawdown from the running peak, three ways.

In [ ]:
by_day = curve.groupby(curve.index.normalize())
low, high = by_day["adverse"].min(), by_day["equity"].max()
balance = result["aum"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38], vertical_spacing=0.06,
                    subplot_titles=("Account equity (R$)", "Drawdown from the running peak (R$)"))
fig.add_trace(go.Scatter(x=low.index, y=low, line=dict(width=0), showlegend=False, hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(x=high.index, y=high, line=dict(width=0), fill="tonexty", fillcolor="rgba(120,120,120,0.25)",
                         name="intraday range"), row=1, col=1)
fig.add_trace(go.Scatter(x=balance.index, y=balance, name="balance (end of day)", line=dict(width=2)), row=1, col=1)

peak_balance = np.maximum.accumulate(np.concatenate(([config.capital], balance.to_numpy())))[1:]
peak_close = np.maximum.accumulate(np.concatenate(([config.capital], by_day["equity"].last().to_numpy())))[1:]
fig.add_trace(go.Scatter(x=balance.index, y=-(peak_balance - balance), name="balance drawdown"), row=2, col=1)
fig.add_trace(go.Scatter(x=low.index, y=-(peak_close - low), name="adverse drawdown (worst of the day)"), row=2, col=1)
fig.update_layout(height=620, legend_orientation="h", margin=dict(t=50, b=20))
fig.show()

## 6. Daily results

The distribution of the daily result and the extremes. `ret` is the net result over the equity of the day before.

In [ ]:
fig = go.Figure(go.Histogram(x=traded["net"], nbinsx=60))
fig.update_layout(title="Daily net result (R$)", height=320, margin=dict(t=50, b=20), bargap=0.05)
fig.show()

view = traded[["contracts", "trades", "gross", "cost", "net", "ret"]].assign(ret=lambda t: t["ret"] * 100)
print("Ten worst days")
display(view.nsmallest(10, "net"))
print("Ten best days")
display(view.nlargest(10, "net"))

In [ ]:
net = traded["net"]
monthly = net.groupby([net.index.year, net.index.month]).sum().unstack(0).reindex(range(1, 13))
monthly.index = pd.Index([f"{m:02d}" for m in monthly.index], name="month")
yearly = pd.DataFrame({"net R$": net.groupby(net.index.year).sum(), "days": net.groupby(net.index.year).size(),
                       "worst day": net.groupby(net.index.year).min(), "contracts (avg)": traded["contracts"].groupby(traded.index.year).mean()})
display(yearly)
display(monthly.round(0))

## 7. Where in the day the result is made

Points per contract by 30-minute bucket, for the version with **no exit time** (so the exit you chose above does not hide anything). A *diagnostic*: it showed that almost everything was earned after 17:00 on days when the trend survived every stop. Positions only exist on the days the band and VWAP rule kept them, so this is not "what the market does at 17:30".

In [ ]:
free = replace(config, exit_minute=minutes("24:00"))
exposure = strategy_exposure(bars, free)
step = bars["close"].groupby(day).diff()
points = (exposure * step).fillna(0.0)
minute = bars.index.hour * 60 + bars.index.minute
bucket = pd.Series((minute // 30) * 30, index=bars.index)
by_bucket = points.groupby(bucket).sum()
tab = pd.DataFrame({"points per contract": by_bucket, "share of total, %": by_bucket / points.sum() * 100})
tab.index = [f"{m // 60:02d}:{m % 60:02d}" for m in tab.index]

fig = go.Figure(go.Bar(x=tab.index, y=tab["points per contract"]))
fig.update_layout(title="Points per contract by 30-minute bucket (no exit time)", height=320, margin=dict(t=50, b=20))
fig.show()
display(tab)

## 8. One day in detail

A dropdown over the ten best and ten worst days of the run above, plus `show_day("YYYY-MM-DD")` for any day. Lines: the close, the Noise Area bands, the VWAP, the bars where a position is held, and the intraday equity (adverse equity dotted). Candles are never drawn.

In [ ]:
sigma = noise_sigma(bars[["open", "close"]], config.lookback, config.min_days)
bands = noise_bands(bars, sigma, config.band_mult)
vwap = session_vwap(bars)
held = strategy_exposure(bars, config)


def day_traces(date: str) -> list:
    sel = day == pd.Timestamp(date)
    t = bars.index[sel]
    in_position = held[sel] != 0
    close = bars.loc[sel, "close"]
    return [
        go.Scatter(x=t, y=close, name="close", line=dict(color="#1f77b4", width=1.5), xaxis="x", yaxis="y"),
        go.Scatter(x=t, y=bands.loc[sel, "ub"], name="upper band", line=dict(color="#2ca02c", dash="dash"), xaxis="x", yaxis="y"),
        go.Scatter(x=t, y=bands.loc[sel, "lb"], name="lower band", line=dict(color="#d62728", dash="dash"), xaxis="x", yaxis="y"),
        go.Scatter(x=t, y=vwap[sel], name="VWAP", line=dict(color="#ff7f0e"), xaxis="x", yaxis="y"),
        go.Scatter(x=t[in_position], y=close[in_position], name="position held", mode="markers",
                   marker=dict(size=4, color=np.where(held[sel][in_position] > 0, "#2ca02c", "#d62728")), xaxis="x", yaxis="y"),
        go.Scatter(x=t, y=curve.loc[sel, "equity"], name="equity", line=dict(color="#1f77b4"), xaxis="x2", yaxis="y2"),
        go.Scatter(x=t, y=curve.loc[sel, "adverse"], name="adverse equity", line=dict(color="#1f77b4", dash="dot"), xaxis="x2", yaxis="y2"),
    ]


def show_day(date: str) -> None:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.05)
    for trace in day_traces(date):
        row = 2 if trace.yaxis == "y2" else 1
        trace.update(xaxis=None, yaxis=None)
        fig.add_trace(trace, row=row, col=1)
    row_ = result.loc[date]
    fig.update_layout(height=620, margin=dict(t=60, b=20), legend_orientation="h",
                      title=f"{date}: {int(row_['contracts'])} contracts, net R$ {row_['net']:,.2f}")
    fig.show()


examples = list(traded["net"].nsmallest(10).index) + list(traded["net"].nlargest(10).index)
labels = [d.strftime("%Y-%m-%d") for d in examples]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.05)
per_day = 7
for i, date in enumerate(labels):
    for trace in day_traces(date):
        row = 2 if trace.yaxis == "y2" else 1
        trace.update(xaxis=None, yaxis=None, visible=(i == 0))
        fig.add_trace(trace, row=row, col=1)
buttons = []
for i, date in enumerate(labels):
    visible = [j // per_day == i for j in range(len(labels) * per_day)]
    net_ = result.loc[date, "net"]
    buttons.append(dict(label=f"{date}  R$ {net_:,.0f}", method="update",
                        args=[{"visible": visible}, {"title": f"{date}: net R$ {net_:,.2f}"}]))
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0.0, xanchor="left", y=1.15, yanchor="top")], height=640,
                  margin=dict(t=90, b=20), legend_orientation="h", title=f"{labels[0]}: net R$ {result.loc[labels[0], 'net']:,.2f}")
fig.show()

## 9. Costs

The same run with 0 to 4 ticks of slippage per contract and side (a tick is R$ 1.00; the B3 fee is R$ 0.25 a side). The edge per contract is small, so this is the table to read before believing any profit.

In [ ]:
rows = {}
for ticks in (0, 0.5, 1, 2, 3, 4):
    rows[f"{ticks:g} ticks"] = evaluate(make_config(fee_per_side=B3_FEE_PER_SIDE + ticks * TICK_REAIS))
display(pd.DataFrame(rows).T[["net R$", "R$ / year", "Sharpe", "t", "balance DD", "adverse DD", "worst day"]].round(2))

## 10. Where this leaves the analysis

- The final test period has **not** been read. Before it is, the owner approves the pass rule and the variants to pre-register (`docs/brief.md` section 10), and the spec and code are frozen at a git tag.
- Open questions that change the numbers: B3's closing rules for WIN, what slippage a small market order really pays, the roll-day gap in the unadjusted series (it moved the result little on development data), and the first-day boundary quirk of the authors' notebook.